# Tutorial 1: Your First Compliance Contract

This notebook walks through the core comfio workflow:

1. Load sensor data into `SensorData`
2. Evaluate each domain (thermal, visual, acoustic, IAQ)
3. Combine into a Global IEQ Index
4. Check compliance and generate a contract-ready report

In [ ]:
import numpy as np
import pandas as pd
from comfio import (
    SensorData, evaluate_thermal, evaluate_visual,
    evaluate_acoustic, evaluate_iaq,
    calculate_global_ieq, calculate_compliance,
)

## 1. Create Synthetic Sensor Data

In [ ]:
rng = np.random.default_rng(42)
n = 100

df = pd.DataFrame({
    'timestamp': pd.date_range('2024-06-01', periods=n, freq='5min'),
    'air_temp_c': rng.normal(24.0, 1.5, n),
    'radiant_temp_c': rng.normal(24.0, 1.0, n),
    'air_velocity_ms': rng.normal(0.1, 0.02, n),
    'relative_humidity_pct': rng.normal(50.0, 5.0, n),
    'illuminance_lux': rng.normal(500.0, 50.0, n),
    'noise_laeq_db': rng.normal(40.0, 5.0, n),
    'co2_ppm': rng.normal(800.0, 100.0, n),
})

sd = SensorData(df)
print(f'Loaded {len(df)} timestamps')
print(f'Available domains: {sd.available_domains()}')

## 2. Evaluate Each Domain

In [ ]:
thermal = evaluate_thermal(
    tdb=sd.get_column('air_temp_c'),
    tr=sd.get_column('radiant_temp_c'),
    vr=sd.get_column('air_velocity_ms'),
    rh=sd.get_column('relative_humidity_pct'),
    met=1.2, clo=0.5,
)
print(f'Thermal — Mean PMV: {np.mean(thermal.pmv):.2f}, Mean PPD: {np.mean(thermal.ppd):.1f}%')

visual = evaluate_visual(illuminance=sd.get_column('illuminance_lux'))
print(f'Visual — Mean score: {np.mean(visual.score):.1f}/100')

acoustic = evaluate_acoustic(laeq=sd.get_column('noise_laeq_db'))
print(f'Acoustic — Mean score: {np.mean(acoustic.score):.1f}/100')

iaq = evaluate_iaq(co2=sd.get_column('co2_ppm'))
print(f'IAQ — Mean score: {np.mean(iaq.score):.1f}/100')

## 3. Calculate Global IEQ Index

In [ ]:
ieq = calculate_global_ieq(
    thermal=thermal, visual=visual,
    acoustic=acoustic, iaq=iaq,
)
print(f'Global IEQ Index — Mean: {np.mean(ieq.index):.1f}/100')
print(f'Min: {np.min(ieq.index):.1f}, Max: {np.max(ieq.index):.1f}')
print(f'Domains: {ieq.domains}')
print(f'Weights: {ieq.weights_used}')

## 4. Check Compliance

In [ ]:
report = calculate_compliance(
    ieq_result=ieq,
    threshold=80.0,
    period_start=1717200000.0,
    period_end=1717286400.0,
)
print(f'Compliance rate: {report.compliance_rate:.1%}')
print(f'Is compliant: {report.is_compliant}')
print(f'Mean IEQ: {report.mean_ieq:.1f}')

## 5. Export Contract-Ready JSON

In [ ]:
json_output = report.to_contract_json()
print(json_output)